In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import glob
from sklearn.model_selection import train_test_split
from transformers import AutoFeatureExtractor, ASTForAudioClassification
import random
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

np.random.seed(42)
torch.manual_seed(42)
INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
MASHUPS_PATH = os.path.join(INPUT_BASE, 'mashups')
TEST_CSV = os.path.join(INPUT_BASE, 'test.csv')

GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

SAMPLE_RATE = 16000
DURATION = 10
TARGET_LENGTH = SAMPLE_RATE * DURATION

MODEL_CHECKPOINT = "MIT/ast-finetuned-audioset-10-10-0.4593"

#### Q1) Exactly how many recipe items will be allocated to your validation set (val_recipes) list?
#### Ans: `200`

In [3]:
def create_recipes(genres, recipes_per_genre=100):
    all_recipes = []
    for genre in genres:
        genre_path = os.path.join(STEMS_PATH, genre)
        song_folders = glob.glob(os.path.join(genre_path, '*'))
        
        for i in range(recipes_per_genre):
            recipe = {
                'genre': genre,
                'genre_idx': genres.index(genre),
                'song_folders': song_folders,
                'recipe_id': i
            }
            all_recipes.append(recipe)
    
    return all_recipes

recipes = create_recipes(GENRES, recipes_per_genre=100)
print(f"Total recipes: {len(recipes)}")

train_recipes, val_recipes = train_test_split(
    recipes,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print(f"Train recipes: {len(train_recipes)}")
print(f"Validation recipes: {len(val_recipes)}")

Total recipes: 1000
Train recipes: 800
Validation recipes: 200


#### Q2) Before this final mix array is passed to the feature extractor, what must its exact 1D NumPy shape be?
#### Ans: `(160000,)`

In [4]:
stem1 = np.random.randn(TARGET_LENGTH)
stem2 = np.random.randn(TARGET_LENGTH)
stem3 = np.random.randn(TARGET_LENGTH)
stem4 = np.random.randn(TARGET_LENGTH)
noise = np.random.randn(TARGET_LENGTH)

mix = stem1 + stem2 + stem3 + stem4

intensity = 0.2
final_mix = mix + (noise * intensity)

print(f"Final mix shape: {final_mix.shape}")

Final mix shape: (160000,)


#### Q3) What is the exact shape of the resulting PyTorch tensor? (This is the spectrogram matrix the AST model will process).
#### Ans: `[1024, 128]`

In [5]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)

mix = np.ones(160000)
inputs = feature_extractor(
    mix,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs['input_values'].squeeze(0)

print(f"Input values shape after squeeze(0): {input_values.shape}")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Input values shape after squeeze(0): torch.Size([1024, 128])


#### Q4) Calculate the number of trainable parameters in this newly configured model. What is the exact integer value?
#### Ans: `86196490`

In [6]:
model = ASTForAudioClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=10,
    ignore_mismatched_sizes=True
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 86,196,490
Trainable parameters: 86,196,490


#### Q5) What is the resulting value at index 0 of the normalized array, rounded to 3 decimal places?
#### Ans: `-1.000`

In [7]:
y_test = np.array([-0.85, 0.40, 0.20, -0.10])
y_normalized = y_test / (np.max(np.abs(y_test)) + 1e-9)
print(f"Normalized array: {y_normalized}")

print(f"Value at index 0: {round(y_normalized[0], 3)}")

Normalized array: [-1.          0.47058823  0.23529412 -0.11764706]
Value at index 0: -1.0
